# Assignment 2: Construct and train progressive EO models

Do not copy dataloader code into this notebook. Import the A1 module and
reconstruct the exact saved membership from `data_protocol.json`.

| Task | Student work | Evidence |
|---|---|---|
| Reuse A1 | rebuild exact loaders | protocol and membership equality |
| MNIST | adapt ResNet18 for 1×28×28 inputs | logits, history, validation checkpoint |
| EuroSAT | implement 13-band classifier | architecture/shape table and curves |
| Planet | implement five-stage, three-class U-Net | five skips, bottleneck, diagram |
| Training | complete full train/validation loops | histories and best checkpoints |


## Grading rubric (100 points)

| Assessed evidence | Points | Full-credit standard |
|---|---:|---|
| Exact reconstruction of A1 loaders/protocol | 10 | MNIST, EuroSAT100, and Planet membership equality asserted; no copied loader definitions |
| Adapted MNIST ResNet18 | 12 | One-channel input, ten raw logits, measured shape check, configuration saved |
| EuroSAT classifier | 10 | Thirteen-channel input, ten raw logits, measured shapes and architecture explanation |
| Five-stage Planet U-Net | 25 | Five encoders/downsamplings, separate bottleneck, five decoders/skips, three logits, even/odd size preservation |
| Training and validation implementation | 25 | Every batch processed; correct device/grad/mode handling; observation-weighted losses; multiclass metrics; validation-selected checkpoints |
| Histories, diagrams, shape tables, hashes | 8 | CSVs, curves, architecture diagram, measured table, configurations and hashes saved |
| Critical understanding answers | 10 | Correct reasoning about logits, receptive field, skips, imbalance, modes, validation, and checkpoints |
| **Total** | **100** | |

Code receives credit only with verification evidence. Hard-coded shapes, test-set
checkpoint selection, a two-stage U-Net, or binary Planet output lose the
corresponding rubric credit even when the notebook reaches its final cell.


## Where you write code in the modular assignment

The notebooks are the **orchestrators**, not the main implementation files. Keep
them open to read instructions, reload your modules, run checks, make figures, and
write interpretations. Write assessed Python implementations in these exact files:

| Assignment | File you edit | Symbols you implement |
|---|---|---|
| A1 | `src/student_pipeline.py` | `paired_flip`, `normalize_image`, `training_transform`, `evaluation_transform`, `build_mnist_loaders`, `build_eurosat_loaders`, `PlanetCatalogDataset`, `match_directory_pairs`, `PlanetDirectoryDataset`, `build_planet_loaders`, `save_data_protocol`, `load_data_protocol` |
| A2 | `src/student_models.py` | `MNISTResNet18`, `EuroSATClassifier`, `PlanetUNet` |
| A2 | `src/student_training.py` | `run_epoch`, `fit` |
| A3 | `src/student_inference.py` | `predict_classification`, `export_prediction`, `predict_and_export` |

Replace the `TODO`/`NotImplementedError` bodies while preserving the public
function names, arguments, return contracts, and class attributes described in
their docstrings. Save the module, rerun the notebook's reload cell, and recreate
any datasets, loaders, models, or optimizers that were built from the old code.

You also write explanatory work in the notebook's clearly labelled evidence,
architecture, interpretation, and critical-question Markdown cells, and document
agent assistance in `AI_USE.md`. Do **not** implement assessed work in notebook
scratch cells or edit `assignment_setup.py`, `student_checks.py`, `assessment_support.py`,
the catalog, or the tests. Those supplied files define setup, checks, and evidence
contracts and should remain unchanged.


## Foundations: from batches to trainable models

A model's final dimension describes the prediction problem. Image classifiers
return `[B,C]`; the Planet segmenter returns `[B,C,H,W]`. These are **raw logits**:
unbounded scores that cross entropy converts internally with log-softmax. Applying
softmax before `CrossEntropyLoss` changes the optimization problem and should not
be done here.

The MNIST ResNet18 exercise shows how to adapt a supplied architecture: its first
convolution must accept one channel, the early max-pool is unnecessarily aggressive
for 28×28 digits, and its final linear layer must produce ten logits. EuroSAT
requires thirteen input channels. Neither modification changes the meaning of a
batch target: it remains one integer class per image.

The Planet U-Net is symmetric but not shallow. Each of five encoder blocks creates
a skip tensor **before** pooling. The separate bottleneck operates after the fifth
downsampling, at about 1/32 input resolution. Five decoder stages progressively
upsample and fuse matching skips. Deep features carry broader context; skips restore
location and boundary detail. Odd image sizes require explicit alignment to the
measured skip size rather than assumptions based on exact powers of two.

One epoch means every batch in one loader has been processed once. Training uses
`train()`, gradients, `zero_grad`, `backward`, and `step`. Validation uses `eval()`
and `no_grad()` with no optimizer update. Average loss must be weighted by the
number of samples or valid pixels; an unweighted average of unequal batch means is
biased. The minimum validation loss selects a checkpoint. The test split is not a
checkpoint-selection tool.


## Setup: upload the complete folder

**2i2c is the primary platform.** Upload or extract the entire folder as
`/home/jovyan/assignments`, then open this notebook from the folder root. Keep
`src/`, `catalog.csv`, all notebooks, and `outputs/` together.
The `/home/jovyan` directory is persistent across normal 2i2c sessions.

The setup below adds `PROJECT_ROOT/src` directly to Python's import path. It does
not create an environment or install the assignment as a package. It first checks
the modules already supplied by the 2i2c image. Only if that check reports a
missing package should you open a terminal in `assignments` and run:

```bash
python -m pip install -r requirements.txt
```

**Colab fallback:** put the complete folder in
`My Drive/IDLEO/assignments`, open the notebook in Colab, and run the same
cells. Code, protocols, outputs, and checkpoints remain in Drive. Runtime data
may need to be downloaded again after a reset.


In [ ]:
import importlib
import json  # noqa: F401
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/IDLEO/assignments")
else:
    preferred = Path("/home/jovyan/assignments")
    if preferred.is_dir():
        PROJECT_ROOT = preferred
    else:
        working_directory = Path.cwd().resolve()
        PROJECT_ROOT = working_directory

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from assignment_setup import check_dependencies, prepare_project  # noqa: E402

paths = prepare_project(PROJECT_ROOT)
versions = check_dependencies()
device_name = "cuda" if __import__("torch").cuda.is_available() else "cpu"
print("Project:", paths.project_root)
print("Data:", paths.data_root)
print("Device:", device_name)
print("Dependencies:", versions)

## Agentic coding without outsourcing the assignment

An agent may help you plan, inspect an error, propose a test, or review one saved
function. Work on one named TODO at a time. Give the function contract, shapes,
and the failing check; ask for an explanation of the proposed change. Read every
line, reject unsupported assumptions, save the file yourself, reload it, and run
both the supplied check and an independent counterexample. Do not ask an agent to
"complete the assignment" or paste an answer repository. Record what you accepted,
changed, rejected, and verified in `AI_USE.md`. You must be able to explain and
modify the final code without assistance.


## Reload the saved A1 and A2 source modules


In [ ]:
import matplotlib.pyplot as plt
import torch

import student_checks as checks
import student_models as models
import student_pipeline as pipeline
import student_training as training
from assessment_support import set_seed, sha256

pipeline = importlib.reload(pipeline)
models = importlib.reload(models)
training = importlib.reload(training)
set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Task 1: Reconstruct A1 data exactly (10 points)

Load the saved protocol and pass its IDs into the A1 constructors. Assert
exact equality before training. Repairing A1 preprocessing creates a new
experiment: rerun A1 and save a new protocol rather than silently changing it.


In [ ]:
protocol_path = paths.output_root / "assignment_1/data_protocol.json"
protocol = pipeline.load_data_protocol(protocol_path)
if "mnist_split_ids" not in protocol:
    raise ValueError("A1 protocol predates the required MNIST track; rerun A1")
mnist = pipeline.build_mnist_loaders(
    paths.data_root / "mnist",
    batch_size=protocol["settings"]["mnist_batch_size"],
    seed=protocol["seed"],
    split_ids=protocol["mnist_split_ids"],
    download=True,
)
eurosat = pipeline.build_eurosat_loaders(
    paths.data_root / "eurosat",
    batch_size=protocol["settings"]["euro_batch_size"],
    seed=protocol["seed"],
    split_ids=protocol["eurosat_split_ids"],
    download=True,
)
planet = pipeline.build_planet_loaders(
    paths.catalog_path,
    paths.data_root,
    batch_size=protocol["settings"]["planet_batch_size"],
    seed=protocol["seed"],
    selected_rows=protocol["planet_selected_rows"],
)
assert mnist["split_ids"] == protocol["mnist_split_ids"]
assert eurosat["split_ids"] == protocol["eurosat_split_ids"]
assert planet["selected_rows"] == protocol["planet_selected_rows"]

## Task 2: Adapt ResNet18 for MNIST (12 points)

Implement `MNISTResNet18` in `student_models.py` using
`torchvision.models.resnet18(weights=None)`. Replace the first convolution
so it accepts one channel, replace the early max-pool with `Identity` for
small 28×28 inputs, and replace the final layer with ten logits. Do not
download pretrained weights. Record the input, intermediate/output shape,
trainable parameter count, and why logits—not probabilities—are returned.


In [ ]:
mnist_config = {"num_classes": 10}
mnist_classifier = models.MNISTResNet18(**mnist_config).to(device)
mnist_images, mnist_targets = next(iter(mnist["train"]))
mnist_logits = mnist_classifier(mnist_images.to(device))
print("MNIST:", mnist_images.shape, mnist_logits.shape)
print(
    "parameters:",
    sum(parameter.numel() for parameter in mnist_classifier.parameters()),
)
assert mnist_logits.shape == (mnist_images.shape[0], 10)

**Evidence to submit (12 points):** the adapted layers, measured shape and
parameter count, and a concise explanation of why an unchanged three-channel
ImageNet stem is incompatible with MNIST.


## Task 3: EuroSAT classifier (10 points)

Implement `EuroSATClassifier` in `student_models.py`. It accepts 13-band
`[B,13,H,W]` tensors and returns ten raw logits `[B,10]`. Draw the model and
record measured feature shapes rather than guessing them.


In [ ]:
classifier_config = {"in_channels": 13, "num_classes": 10, "base_channels": 32}
classifier = models.EuroSATClassifier(**classifier_config).to(device)
euro_images, euro_targets = next(iter(eurosat["train"]))
print("EuroSAT:", euro_images.shape, classifier(euro_images.to(device)).shape)

## Task 4: Five-stage Planet U-Net (25 points)

Implement `PlanetUNet`, not the earlier two-stage teaching baseline. It must
expose five encoder stages, a separate bottleneck, five decoder stages, and
five corresponding skip connections. A four-band input returns three raw
class logits per pixel. Preserve even and odd input sizes. Submit a readable
architecture diagram labelling stages, skips, channels, and spatial sizes.


In [ ]:
unet_config = {"in_channels": 4, "num_classes": 3, "base_channels": 8}
segmenter = models.PlanetUNet(**unet_config).to(device)
checks.check_unet_architecture(segmenter.cpu())
segmenter = segmenter.to(device)
planet_images, planet_targets, _ = next(iter(planet["train"]))
planet_logits = segmenter(planet_images.to(device))
print("Planet:", planet_images.shape, planet_logits.shape, planet_targets.shape)
assert planet_logits.shape == (
    planet_images.shape[0],
    3,
    *planet_targets.shape[-2:],
)

**Architecture and evidence (8 points):** add your own diagram and a measured table showing
all five encoder outputs, the 1/32-resolution bottleneck, and five decoder
outputs. Explain how each skip is aligned for odd-sized inputs.


## Task 5: Full training and validation loops (25 points)

Implement `run_epoch` and `fit` in `student_training.py`. You own every batch
iteration, device transfer, gradient operation, validation context, loss
aggregation, global confusion count, history row, and best-validation-loss
checkpoint. Segmentation uses three-class cross entropy with target `-1`
ignored. Test loaders are prohibited here.


In [ ]:
checks.check_training_loop(training, device)
EPOCHS = 3
MNIST_EPOCHS = 2

mnist_optimizer = torch.optim.Adam(mnist_classifier.parameters(), lr=1e-3)
mnist_checkpoint = paths.output_root / "assignment_2/mnist_resnet18.pth"
mnist_history = training.fit(
    mnist_classifier,
    mnist["train"],
    mnist["validate"],
    device,
    "classification",
    mnist_optimizer,
    MNIST_EPOCHS,
    mnist_checkpoint,
    mnist_config,
)
mnist_history.to_csv(
    paths.output_root / "assignment_2/mnist_history.csv", index=False
)

classifier_optimizer = torch.optim.Adam(classifier.parameters(), lr=1e-3)
classifier_checkpoint = (
    paths.output_root / "assignment_2/eurosat_classifier.pth"
)
classifier_history = training.fit(
    classifier,
    eurosat["train"],
    eurosat["validate"],
    device,
    "classification",
    classifier_optimizer,
    EPOCHS,
    classifier_checkpoint,
    classifier_config,
)
classifier_history.to_csv(
    paths.output_root / "assignment_2/eurosat_history.csv", index=False
)

segmentation_optimizer = torch.optim.Adam(segmenter.parameters(), lr=1e-3)
segmentation_checkpoint = paths.output_root / "assignment_2/planet_unet.pth"
segmentation_history = training.fit(
    segmenter,
    planet["train"],
    planet["validate"],
    device,
    "segmentation",
    segmentation_optimizer,
    EPOCHS,
    segmentation_checkpoint,
    unet_config,
)
segmentation_history.to_csv(
    paths.output_root / "assignment_2/planet_history.csv", index=False
)
display(mnist_history, classifier_history, segmentation_history)

figure, axes = plt.subplots(1, 3, figsize=(15, 4))
for axis, history, title in zip(
    axes,
    (mnist_history, classifier_history, segmentation_history),
    ("MNIST", "EuroSAT", "Planet"),
):
    history.plot(x="epoch", y=["train_loss", "validation_loss"], ax=axis)
    axis.set_title(title)
figure.tight_layout()
figure.savefig(paths.output_root / "assignment_2/training_curves.png", dpi=150)

**Experiment evidence:** plot train/validation loss, report wall time and
budgets, compare against majority/noncrop baselines, and interpret per-class
support and IoU. Diagnose a failed or inconclusive experiment honestly; a
declining loss alone does not establish useful boundary segmentation.


## Save the progressive handoff manifest


In [ ]:
manifest = {
    "data_protocol_sha256": sha256(protocol_path),
    "student_pipeline_sha256": sha256(
        paths.project_root / "src/student_pipeline.py"
    ),
    "student_models_sha256": sha256(
        paths.project_root / "src/student_models.py"
    ),
    "student_training_sha256": sha256(
        paths.project_root / "src/student_training.py"
    ),
    "mnist_checkpoint": mnist_checkpoint.name,
    "mnist_checkpoint_sha256": sha256(mnist_checkpoint),
    "classifier_checkpoint": classifier_checkpoint.name,
    "classifier_checkpoint_sha256": sha256(classifier_checkpoint),
    "segmentation_checkpoint": segmentation_checkpoint.name,
    "segmentation_checkpoint_sha256": sha256(segmentation_checkpoint),
    "mnist_config": mnist_config,
    "classifier_config": classifier_config,
    "unet_config": unet_config,
}
manifest_path = paths.output_root / "assignment_2/experiment_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(manifest_path.read_text())

## Critical understanding questions (10 points)

1. Why does A2 import and reconstruct A1 loaders instead of copying their code?
2. Why are ResNet18's input convolution and early max-pool both reconsidered for 28×28 one-channel digits?
3. At which resolutions are the five U-Net skip tensors created, and why are they taken before pooling?
4. What does the bottleneck add beyond simply using five encoder blocks?
5. Why must the final Planet tensor be `[B,3,H,W]` rather than `[B,H,W]` or `[B,1,H,W]`?
6. Why does cross entropy receive raw logits rather than softmax probabilities?
7. How can background-dominated pixel accuracy conceal failure on class `2` boundaries?
8. What state changes between `model.train()` and `model.eval()`, even when gradients are disabled?
9. Why must the checkpoint be selected by validation loss rather than the final epoch or test score?
10. How would you detect an incorrectly weighted mean of unequal batch losses?
